# Property-based testing of an ELM process with `spel proptest`
### Case study: `CanopyFluxes`

This notebook walks through testing one ELM routine, `CanopyFluxesMod::CanopyFluxes`, with
**properties** (invariants that must hold for every input), **metamorphic relations** (how the
outputs must change when the inputs change in a known way) and **differential checks** (two builds
of the routine must agree). It uses SPEL's unit-test extraction,
Hypothesis and SPEL's Fortran parser. By the end you should be able to do the same for another
subroutine; Part 14 is the checklist for that.

```
 E3SM source ──spel analyze/create──▶ unit-test case (fut.pkl, Fortran sources)
                                            │
                        spel proptest build │  bind(C) shim + libspelcase.so
                                            ▼
 <name>.proptest.toml ──spec.load──▶ inventory + scenarios + checks
   (Fortran syntax)                         │
                                            ▼
     Hypothesis draws inputs ─▶ worker process calls the Fortran routine ─▶ numpy evaluates
     (one draw = every element)  on a synthetic domain (one call per draw)     the checks per element
```

Some principles of the design are useful to keep in mind:

* **Nothing is read from the source by text.** The variables, their dimensions, read/write
  status, namelist guards and associate names all come from SPEL's analysis of the case. The
  checks are Fortran expressions parsed by SPEL's own parser, so you write `t_veg(p)` exactly as
  CanopyFluxes does.
* **Inputs are synthetic.** A *scenario* gives a rule (range, distribution, values, ...) for
  every input the routine reads. Reference data from a real ELM run only *helps you write* those
  rules (Part 3); it never feeds a test. An input without a rule stops the run: nothing is
  defaulted without a word.
* **The spec is read under its namelist options.** A variable that is only used in branches the
  options turn off (the C13 fields with `use_c13` off) needs no rule and cannot be named in a
  check. Another setting of an option can be the follow-up of a relation (Part 9).
* **The routine runs in a separate process.** A Fortran `stop`, `endrun` or segfault becomes a
  failing example, which Hypothesis shrinks like any other.

**Contents.**
0. Setup.
1. Build the case as a Python-callable library.
2. The inventory: what the analysis knows about the routine's variables.
3. Reference statistics: a starting point for input rules.
4. A first spec: scenarios and missing rules.
5. Expressions in Fortran syntax.
6. One example by hand: run the routine and look at its outputs.
7. Properties, and a failure that is the scenario's fault.
8. Metamorphic relations, and a failure that is a real finding.
9. Namelist options: another setting as the follow-up.
10. Coverage: does the scenario reach the code?
11. Differential checks: two builds, one answer.
12. Active subspace: which inputs matter.
13. The same from the command line.
14. Checklist for your own subroutine.

## 0. Setup

Prerequisites (once):

```bash
pip install -e '.[proptest]'                          # SPEL plus Hypothesis
spel create -s canopyfluxes -c canflux --run-case    # an ordinary unit-test case and its reference capture
```

You also need a reference capture of the routine, used here only for statistics, constants and
level extents (README: "Capture reference data from a real ELM run"). This notebook uses
`unit-tests/input-data/canflux`.

For your own routine, **change the two names in the next cell**. What should outlive the case
(rule files, this tutorial's spec, failing examples) goes under `unit-tests/proptest-data/<CASE>/`.
Scratch (synthetic domains, logs) goes under `unit-tests/<CASE>/proptest/`, which `spel create`
deletes with the case.

`spel proptest init <CASE>` writes a starter spec in one step. This notebook builds one by hand
to show each part; the format is described in `spel/proptest/SPEC.md`.

In [1]:
from pathlib import Path
import textwrap

from collections import Counter

import numpy as np
import pandas as pd

from spel.scripts.config import unittests_dir
from spel.proptest import build, coverage, paths, spec, stats, subspace, runner
from spel.proptest.domain import CONSTANTS_FILE, read_constants
from spel.proptest.inventory import Inventory
from spel.proptest.fexpr import uses

CASE = "canflux"           # unit-tests/<CASE>, made by `spel create`
REFERENCE = "canflux"      # unit-tests/input-data/<REFERENCE>, a capture of the same routine

case_dir = Path(unittests_dir) / CASE
reference_dir = case_dir.parent / "input-data" / REFERENCE
work = paths.data_dir(case_dir) / "tutorial"      # unit-tests/proptest-data/<CASE>/tutorial
work.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)
print(case_dir, reference_dir, work, sep="\n")

/home/me/climate-projects/SPEL/unit-tests/canflux
/home/me/climate-projects/SPEL/unit-tests/input-data/canflux
/home/me/climate-projects/SPEL/unit-tests/proptest-data/canflux/tutorial


## 1. Build the case as a library

`spel proptest build` generates `SpelPyMod.F90`, a `bind(C)` module with five entry points:
`spel_py_init` (reads the synthetic domain), `spel_py_compute` (calls the routine),
`spel_py_view` (the address and bounds of any variable, so Python reads and writes the Fortran
memory directly), `spel_py_trap` and `spel_py_true`. It then builds the case with CMake as
`build-py/libspelcase.so`. The routine call and its actual arguments are the same as in the
case's `main.F90`; they come from the analysis (`driver_bindings`), not from text.

The first build compiles the whole case. After that you rarely build yourself: every command
checks whether the case's Fortran changed since the last build and recompiles what changed.

In [2]:
lib = build.build(case_dir, quiet=True)
print(lib)
shim = (case_dir / "SpelPyMod.F90").read_text().splitlines()
print(f"SpelPyMod.F90: {len(shim)} lines; a few of its entry points:")
print("\n".join(l for l in shim if "bind(c" in l.lower())[:600])

/home/me/climate-projects/SPEL/unit-tests/canflux/build-py/libspelcase.so
SpelPyMod.F90: 2583 lines; a few of its entry points:
subroutine spel_py_init(path, n) bind(C, name='spel_py_init')
subroutine spel_py_compute() bind(C, name='spel_py_compute')
subroutine spel_py_trap(on) bind(C, name='spel_py_trap')
function spel_py_true() result(t) bind(C, name='spel_py_true')
function spel_py_view(id, addr, lb, ub) result(rank) bind(C, name='spel_py_view')


The shim comes with `spel_py_manifest.json`, which records for every variable its id in
`spel_py_view`, its type, rank and kind. Python uses it through `ctypes`; you never need to read
it yourself.

## 2. The inventory

`Inventory.load` unpickles the case (`fut.pkl`) and classifies every variable the routine touches:

| kind | meaning | in a scenario |
|---|---|---|
| `topology` | subgrid index fields (`veg_pp%column`, `itype`, ...) | set by the synthetic domain |
| `parameter` | read-only data that is not per element (pftcon arrays, physical constants) | taken from the reference, or ruled |
| `input` | read, never written first | **needs a rule** |
| `inout` | read before its first unconditional write (prognostic state) | **needs a rule** |
| `output` | written before any read | reset to NaN before every call |
| `unused` | captured, but not touched under this configuration | ignored |

The read/write category comes from the same static trace that the SPEL database uses
(`db_export.trace.StaticTrace`). A write *inside an if* doesn't make a variable an output if
it is read later.

In [3]:
inv = Inventory.load(case_dir)
print("routines:", inv.routines)
df = pd.DataFrame([v.summary() for v in inv.vars.values()])
df.groupby("kind").size().to_frame("variables")
print(df)


routines: ['canopyfluxesmod::canopyfluxes']
                             name    kind     type  rank  \
0                     bounds%begg  unused  integer     0   
1                     bounds%endg  unused  integer     0   
2                     bounds%begt  unused  integer     0   
3                     bounds%endt  unused  integer     0   
4                     bounds%begl  unused  integer     0   
..                            ...     ...      ...   ...   
279   energyflux_vars%btran_patch  output     real     1   
280    energyflux_vars%bsun_patch   inout     real     1   
281    energyflux_vars%bsha_patch   inout     real     1   
282  energyflux_vars%btran2_patch  output     real     1   
283  energyflux_vars%rresis_patch   inout     real     2   

                        dims level status  \
0                         []   NaN    NaN   
1                         []   NaN    NaN   
2                         []   NaN    NaN   
3                         []   NaN    NaN   
4         

In [4]:
# A few rows of each kind that matters for a scenario
cols = ["name", "kind", "type", "dims", "status", "units", "guard"]
df[df.kind.isin(["input", "inout", "output"])].groupby("kind").head(4)[cols]

,name,kind,type,dims,status,units,guard
12,cnstate_vars%downreg_patch,input,real,[begp:endp],r,proportion,(.not. use_fates .and. use_c13)
13,cnstate_vars%rc14_atm_patch,input,real,[begp:endp],r,,.not. use_fates
14,photosyns_vars%c3flag_patch,inout,logical,[begp:endp],rw,,.not. use_fates
15,photosyns_vars%ac_patch,inout,real,"[begp:endp, 1:nlevcan]",rw,,(.not. use_hydrstress .and. .not. use_fates)
16,photosyns_vars%aj_patch,inout,real,"[begp:endp, 1:nlevcan]",rw,,(.not. use_hydrstress .and. .not. use_fates)
17,photosyns_vars%ap_patch,inout,real,"[begp:endp, 1:nlevcan]",rw,,(.not. use_hydrstress .and. .not. use_fates)
21,photosyns_vars%vcmax25_top_patch,output,real,[begp:endp],w,umolco2/m^2/s,(.not. use_hydrstress .and. .not. use_fates)
22,photosyns_vars%cp_patch,output,real,[begp:endp],w,,.not. use_fates
23,photosyns_vars%kc_patch,output,real,[begp:endp],w,,.not. use_fates
24,photosyns_vars%ko_patch,output,real,[begp:endp],w,,.not. use_fates


`veg_es%t_veg` is a good example of why the trace matters. CanopyFluxes first writes it inside
`if (frac_veg_nosno == 0)` and later reads it (`tlbef = t_veg`). A classification by first
access would call it an output, reset it to NaN, and every output would come out NaN. The trace
sees the conditional write and calls it `inout`.

The routine refers to most fields through `associate` names. You can use either name in a spec:

In [5]:
print(df.set_index("name").loc["veg_es%t_veg", ["kind", "status", "dims", "units"]].to_dict())
for alias in ["t_veg", "btran", "forc_t", "elai"]:
    print(f"{alias:8s} -> {inv.canonical([alias])}")

{'kind': 'inout', 'status': 'rw', 'dims': ['begp:endp'], 'units': 'k'}
t_veg    -> veg_es%t_veg
btran    -> energyflux_vars%btran_patch
forc_t   -> top_as%tbot
elai     -> canopystate_vars%elai_patch


Which variables matter depends on the **namelist options**. `Inventory.load` counts every branch
of the code as running. A run has a configuration: the options of the reference capture (they
are in its constants file), changed by what the spec sets. `inv.configured(values)` classifies
again from the accesses that run under those values; SPEL's analysis records every `if` that
tests a namelist variable, so it knows which accesses those are.

In [6]:
consts = read_constants(reference_dir / CONSTANTS_FILE)   # the capture's constants, its namelist options among them
consts["soil_stress_method"] = 0                          # CanopyFluxes reads it; this capture does not record it
live = inv.configured(consts)

print("namelist options the routine tests:",
      ", ".join(f"{n} = {inv.fortran_value(n, consts[n])}" for n in sorted(inv.options())))
dead = Counter(", ".join(v.dead_under) for v in live.vars.values() if v.kind == "unused" and v.dead_under)
print(f"{sum(dead.values())} variables are used by no branch that runs under them:", dict(dead.most_common()))
print("photosyns_vars%c13_psnsun_patch:", live.why_unused("photosyns_vars%c13_psnsun_patch"))
pd.DataFrame({"every branch": pd.Series(Counter(v.kind for v in inv.vars.values())),
              "this configuration": pd.Series(Counter(v.kind for v in live.vars.values()))})

namelist options the routine tests: atm_gustiness = .false., force_land_gustiness = .false., implicit_stress = .false., perchroot = .false., perchroot_alt = .false., use_c13 = .false., use_c14 = .false., use_cn = .true., use_fates = .false., use_finetop_rad = .false., use_hydrstress = .false., use_lch4 = .true.
51 variables are used by no branch that runs under them: {'use_fates, use_hydrstress': 34, 'use_c13, use_cn': 8, 'use_c14, use_cn': 3, 'implicit_stress': 2, 'perchroot, perchroot_alt': 2, 'use_c13, use_fates': 1, 'use_finetop_rad': 1}
photosyns_vars%c13_psnsun_patch: it is only used under other namelist options (here use_c13 = .false., use_cn = .true.)


,every branch,this configuration
inout,63,44
input,79,65
output,66,57
parameter,43,34
topology,16,16
unused,17,68


In [7]:
print("filters (clumpfilter members, usable in `over = \"p in <filter>\"`):",
      sorted(k for k, r in inv.filters.items() if r == 1)[:12], "...")
print("some module parameters:", {k: inv.parameters[k] for k in list(inv.parameters)[:8]})

filters (clumpfilter members, usable in `over = "p in <filter>"`): ['do_smb_c', 'hydrologyc', 'hydrononsoic', 'icemecc', 'lakec', 'lakenosnowc', 'lakep', 'lakesnowc', 'natvegp', 'nolakec', 'nolakep', 'nolakeurbanp'] ...
some module parameters: {'shr_kind_cs': 80, 'shr_kind_cm': 160, 'shr_kind_cl': 256, 'shr_kind_cx': 512, 'shr_kind_cxx': 4096, 'iundef': -9999999, 'rundef': -9999999.0, 'fname_len': 256}


## 3. Reference statistics: a starting point for the rules

Writing a rule for each of more than a hundred inputs by hand is tedious. `stats` reads a capture and writes:

* a **rule file** (`<case>.inputs.toml`) with a 1st–99th percentile range per input (a
  `log_range` when it spans many decades, `values` for integers, a constant when it never varies),
  and commented per-pft quantiles;
* a **markdown report** with quantiles, zero and fill-value fractions.

`where` restricts the statistics to elements a condition selects (here, patches with
vegetation). `constants` are the values of Part 2: the capture's, plus `soil_stress_method`, which
it doesn't record. The statistics cover the inputs that are read under these options (`live`).

**The reference is a design aid only.** This capture is winter-like with a tiny LAI, so its
ranges would test a narrow corner of the input space. You'll widen them in Part 7.

In [8]:
res = stats.compute(live, reference_dir, where="frac_veg_nosno(p) > 0", stride=4, constants=consts)
rule_file, report = stats.write(live, res, case_dir)
print(len(res.inputs), "inputs")
print(rule_file, report, sep="\n")

109 inputs
/home/me/climate-projects/SPEL/unit-tests/proptest-data/canflux/stats/canflux.inputs.toml
/home/me/climate-projects/SPEL/unit-tests/proptest-data/canflux/stats/canflux.stats.md


In [9]:
# The rule file: a commented entry per input. Here is one.
text = rule_file.read_text().splitlines()
k = next(i for i, l in enumerate(text) if l.startswith("# canopystate_vars%elai_patch"))
print("\n".join(text[k:k + 9]))

# canopystate_vars%elai_patch(begp:endp) real [m^2/m^2] exposed one-sided leaf area index
#   n=18468 min=0 p50=0.0128403 max=0.0154572 mean=0.00918914 std=0.0064485 zero=29%
#   pft  1: p5..p95 = 0.00657739 .. 0.0147171 (n=3759)
#   pft  7: p5..p95 = 0 .. 0.0148871 (n=3759)
#   pft 13: p5..p95 = 0 .. 0.0148809 (n=3650)
#   pft 14: p5..p95 = 0 .. 0.0148809 (n=3650)
#   pft 15: p5..p95 = 0 .. 0.0148809 (n=3650)
"canopystate_vars%elai_patch" = { range = [0.0, 0.0148956] }



## 4. A first spec

A spec is a TOML file with these tables:

* `[case]`: the unit-test case;
* `[namelist]`: the namelist options of the runs, over the reference's;
* `[domain]`: the synthetic domain. Each element is its own gridcell, column and patch:
  `per_pft` patches of each listed pft, repeated in `nsets` copies. Relations put base and
  follow-up in the two halves of the sets, so `nsets` must be even;
* `[scenario.<name>]`: input rules (`inputs`), rules from files (`extends`), dependent inputs
  (`derive`), filters on drawn examples (`assume`) and the elements to check (`over`);
* `[[property]]` and `[[relation]]`: the checks.

The cells below build the spec in pieces and reload it after each step. `spec.load(path, inv=inv)`
validates every name and expression against the inventory.

In [10]:
spec_path = work / "tutorial.proptest.toml"

HEADER = f"""
[case]
name = "{CASE}"

[settings]
max_examples = 30

[domain]
pfts = [1, 4, 13]          # needleleaf evergreen temperate tree, broadleaf evergreen tropical tree, C3 grass
per_pft = 4
nsets = 4                  # relations: sets 0-1 base, 2-3 follow-up
reference = "{REFERENCE}"  # constants, parameters and level extents

[namelist]
soil_stress_method = 0     # read by CanopyFluxes; the reference does not record it
"""

def write_spec(*parts):
    spec_path.write_text("\n".join(textwrap.dedent(p) for p in parts))
    return spec.load(spec_path, inv=inv)[0]

sp = write_spec(HEADER, """
[scenario.snowfree_day]
over = "p in nolakeurbanp"

[[property]]
name = "outputs are finite"
expr = "finite(t_veg(p))"
""")
scen = sp.scenarios["snowfree_day"]
consts = spec.constants(sp, inv, scen)   # the reference's constants, [domain.constants], [namelist]
live = inv.configured(consts)            # the inventory under the scenario's options
missing = spec.missing_rules(scen, live, consts)
print(len(missing), "inputs have no rule, e.g.", missing[:5])

109 inputs have no rule, e.g. ['canopystate_vars%displa_patch', 'canopystate_vars%dleaf_patch', 'canopystate_vars%elai_patch', 'canopystate_vars%esai_patch', 'canopystate_vars%frac_veg_nosno_patch']


Every input the routine reads under the spec's namelist options needs a rule. A run refuses to start
otherwise, because a silently defaulted input would make the test meaningless. Add the stats
file with `extends` and see what is left:

In [11]:
SCENARIO = f"""
[scenario.snowfree_day]
over = "p in nolakeurbanp"
extends = "{Path("..") / "stats" / rule_file.name}"
"""
PROPS = """
[[property]]
name = "outputs are finite"
expr = "finite(t_veg(p))"
"""
sp = write_spec(HEADER, SCENARIO, PROPS)
missing = spec.missing_rules(sp.scenarios["snowfree_day"], live, consts)
print(missing)
why = {v.name: v.missing for v in res.inputs}
for name in missing:
    print(f"  {name}: {why.get(name) or 'no statistics'}")

[]


Nothing is left. When an input does remain, `stats` says why it has no rule (most often: only
fill values in the reference), and you give it one yourself; a bare value is a constant rule.
Before SPEL read the spec under its namelist options, two inputs remained here: the C13
fractionation factors, which hold only fill values in a run without C13. They are not inputs of
this configuration at all, and a check cannot use what the configuration does not compute:

In [12]:
try:
    write_spec(HEADER, SCENARIO, """
[[property]]
name = "the 13C flux is finite"
expr = "finite(photosyns_vars%c13_psnsun_patch(p))"
""")
except spec.SpecError as exc:
    print("SpecError:", textwrap.fill(str(exc), 110))
sp = write_spec(HEADER, SCENARIO, PROPS)

SpecError: [[property]] 'the 13C flux is finite': photosyns_vars%c13_psnsun_patch has no value in scenario snowfree_day:
it is only used under other namelist options (here use_c13 = .false., use_cn = .true.)


The rule kinds are:

| rule | example |
|---|---|
| constant | `"x" = 1.0` |
| uniform range | `{ range = [0.1, 6.0] }` |
| log-uniform | `{ log_range = [1e-6, 1e-2] }` |
| a set of values | `{ values = [0, 1, 2] }` |
| a distribution | `{ dist = "lognormal", mean = 0.0, sigma = 1.0, min = 0.1, max = 10.0 }` |
| a parameter scaled around its value | `{ scale = [0.8, 1.2] }` |
| per-pft overrides | `{ range = [0.3, 20.0], pft = { 13 = { range = [0.3, 1.0] } } }` |
| per-level overrides | `{ range = [0.1, 0.4], levels = { "1" = 0.3, "11:nlevgrnd" = { range = [0.0, 0.1] } } }` |

Any rule may add `clip = [lo, hi]`.

For every input, Hypothesis draws a *mode*: one value for all elements, independent values for
every entry, edge values (the ends of the range), or for layered inputs independent values per
element with one value down the layers. Shrinking a failing example therefore works on a few
numbers rather than whole arrays.

## 5. Expressions in Fortran syntax

Properties, relations, `derive` and `assume` are Fortran expressions, parsed by SPEL's parser and
evaluated with numpy over all checked elements at once:

* the element variables `g, t, l, c, p` all index the same site, because each element is its own
  gridcell, topounit, landunit, column and patch. So `t_grnd(c)` and `t_veg(p)` can sit in one
  expression;
* the names are the ones CanopyFluxes uses (associate names or `inst%field`), module parameters
  and constants, plus `filter%<member>`;
* in a check a name is the value *after* the run; for a variable the routine reads and writes,
  `input%x` is the value it had before;
* the functions are Fortran's elemental intrinsics and reductions (`abs`, `exp`, `min`, `sum`,
  ...) and these helpers: `finite`, `between(x, lo, hi)` (both ends included),
  `close(a, b, rtol, atol)`, `same(a, b)` (identical, or both NaN), `implies(a, b)`,
  `integral(x, dz)`, `cumsum(x)`, `rel_change(new, old)`. A spec can add its own from a Python
  file (`[functions] modules = ["helpers.py"]`).

Unknown names are an error when the spec loads, not halfway through a run:

In [13]:
names = spec.Names(inv, set(sp.functions))
e = names.expr("between(btran(p), 0.0, 1.0) .and. implies(elai(p) > 0.0, rssun(p) > 0.0)", "demo")
print("uses:", sorted({"%".join(u.path) for u in uses(e.node)}))
try:
    names.expr("t_leaf(p) > 0.0", "demo")
except spec.SpecError as exc:
    print("SpecError:", exc)

uses: ['between', 'btran', 'elai', 'implies', 'p', 'rssun']
SpecError: demo: unknown name 't_leaf' in 't_leaf(p) > 0.0'


## 6. A few examples by hand

Before writing many checks, run the routine a few times and look at what comes out. A
`runner.Session` owns the library, the synthetic domain and the worker process.
`Session.probe` (the command `spel proptest probe`) runs a few examples, the first with every
input at the middle of its rule, and reports what the routine did on the checked elements.
Outputs are reset to NaN before every call, so **an output that stays NaN was never written
there**. That usually means a branch your scenario doesn't reach, or a variable in the wrong
kind; this is the first thing to look at for a new routine.

`Session.inputs_at(scenario, u)` builds one example yourself: every input at quantile `u` of its
rule (a number, or a function such as `rng.random` for independent draws), then `derive`.

In [14]:
from spel.proptest.runner import Session

def per_element(s, arrays, names, elems):
    """Rank-1 subgrid variables at the given (1-based) elements, as a table."""
    out = {"pft": s.layout.pft_of()[elems - 1]}
    for n in names:
        fa = arrays[n]
        if fa.data.ndim == 1:
            out[n.split("%")[-1]] = fa.data[elems - fa.lb[0]]
    return pd.DataFrame(out, index=pd.Index(elems, name="element"))

scen = sp.scenarios["snowfree_day"]
show = ["veg_es%t_veg", "veg_ef%eflx_sh_veg", "veg_wf%qflx_tran_veg", "energyflux_vars%btran_patch",
        "canopystate_vars%elai_patch"]
with Session(sp, inv) as s:
    probed = s.probe(scen, n=4)
    elems = s.elements((scen.level, scen.filter))
    ex = s.inputs_at(scen, 0.5)                       # every input at the middle of its rule
    tab = per_element(s, s.arrays(ex, s.compute(scen, ex, show[:-1])), show, elems)
print(len(elems), "elements in nolakeurbanp")
tab.head(6)

48 elements in nolakeurbanp


,pft,t_veg,eflx_sh_veg,qflx_tran_veg,btran_patch,elai_patch
element,,,,,,
1,1,298.364191,137.254923,0.0,0.0,0.007448
2,1,298.364191,137.254923,0.0,0.0,0.007448
3,1,298.364191,137.254923,0.0,0.0,0.007448
4,1,298.364191,137.254923,0.0,0.0,0.007448
5,4,298.364191,137.254923,0.0,0.0,0.007448
6,4,298.364191,137.254923,0.0,0.0,0.007448


In [15]:
print(probed.text())

scenario snowfree_day: 4 examples (the first at every rule's midpoint), 48 elements, 100% pass `assume`
  101 outputs; never written on the checked elements: 0
  read and written, but never changed:
    canopystate_vars%dleaf_patch
    photosyns_vars%psnsha_wj_patch
    photosyns_vars%psnsha_wp_patch
    veg_wf%irrig_rate
    veg_wf%n_irrig_steps_left
  the same value at every element of every example:
    photosyns_vars%fpsn_wj_patch = 0.0
    photosyns_vars%fpsn_wp_patch = 0.0
    photosyns_vars%mbb_patch = 9.0


Read the probe and the table before going on. Here no output is unwritten. The fields that
would be, the C13 and C14 outputs, belong to branches the namelist options turn off, so they are
not outputs of this configuration and the probe does not list them (Part 2). An output that *is*
listed was reachable and not reached: a branch your scenario misses. That matters when a check
uses it. If *everything* is unwritten, an input is probably classified as an output (Part 2), or
the elements are not where you think they are (`over`). "Read and written, but never changed"
lists state the routine left alone in these examples, and an output with the same value
everywhere often means a branch that is never taken.

## 7. Properties, and failures that are not ELM's fault

Start with properties that must hold whatever the inputs: finite outputs, a soil-water stress
factor in [0, 1], positive and bounded stomatal resistances, and transpiration that is not
negative. `runner.run` gives each check a Hypothesis test and returns one `Outcome` per check.
`max_examples` and `seed` override `[settings]`.

In [ ]:
PROPS = """
[[property]]
name = "outputs are finite"
expr = """ + '"""' + """finite(t_veg(p)) .and. finite(btran(p)) .and. finite(rssun(p)) .and. finite(rssha(p))
          .and. finite(eflx_sh_veg(p)) .and. finite(qflx_tran_veg(p)) .and. finite(ustar(p))""" + '"""' + """

[[property]]
name = "soil water stress in [0, 1]"
expr = "btran(p) >= 0.0 .and. btran(p) <= 1.0"

[[property]]
name = "stomatal resistances in (0, rsmax]"
expr = "rssun(p) > 0.0 .and. rssun(p) <= 2.0e4 .and. rssha(p) > 0.0 .and. rssha(p) <= 2.0e4"

[[property]]
name = "transpiration is not negative"
expr = "qflx_tran_veg(p) >= 0.0"
"""
sp = write_spec(HEADER, SCENARIO, PROPS)

def run(sp, **kw):
    out = runner.run(sp, inv, settings={"seed": 0, **kw.pop("settings", {})}, **kw)
    return pd.DataFrame([(o.scenario, o.check, o.passed) for o in out], columns=["scenario", "check", "passed"]), out

table, outcomes = run(sp)
table

[snowfree_day] property 'outputs are finite' ...
  ok
[snowfree_day] property 'soil water stress in [0, 1]' ...
  ok
[snowfree_day] property 'stomatal resistances in (0, rsmax]' ...


In [ ]:
for o in outcomes:
    if not o.passed:
        lines = o.message.splitlines()
        keep = [l for l in lines if not l.startswith("    ") or any(k in l for k in
                ("rssun", "rssha", "elai", "laisun", "laisha"))]
        print("\n".join(keep))

Read the report before blaming ELM. It gives the failing element (its set and pft), the value
of every name the expression uses, and the drawn inputs at that element. Every failure needs the
same triage: is it the **model**, the **scenario**, or the **property**?

Here every element fails with `rssha = 0`, a shaded stomatal resistance of zero. Look at the
drawn inputs: `laisha_z_patch = 0` (the shaded leaf area of the canopy layer) while
`laisha_patch` (the same quantity, summed) is not, and neither agrees with `elai`. The stats
rules draw each of them independently, so they contradict each other. With no shaded leaf area
in any layer, `Photosynthesis` takes its `else` branch and sets `rs = 0`. ELM is right, and so is
the property for a canopy that has leaves: the *scenario* describes a canopy that cannot exist.

The scenario has more problems, even though no check catches them. The Part 6 table shows
`btran = 0` and no transpiration: the stats ranges describe a winter capture with a leaf area of
0.007, so this scenario hardly tests a canopy at all. Ranges taken from one capture are a
starting point, not a scenario. The rule file also fixes the root fraction at the reference's
value, `rootfr = 0.0666667` in each of 15 layers, which sums to 1.0000005. `btran` is a
root-weighted sum, so once the soil is wet it overshoots 1 by exactly that much. That came up
while the example spec was being written: Hypothesis is good at finding such inconsistencies.

The same property also failed then for a reason that is the *property's* fault: `rssun` came out
as `20000.000000000004`, one rounding error above `rsmax`. `PhotosynthesisMod` caps each canopy
layer's resistance at `rsmax0 = 2e4` and then combines the layers, `rs = laican / gscan - rb`.
When every layer sits at the cap, the result is `rsmax` up to rounding. Comparing floating-point
results exactly with a bound the code reaches is fragile; give the bound a relative tolerance.

The fix is a `derive` rule, which computes an input from other inputs or constants:
`rootfr = 1 / nlevgrnd`. More generally, a scenario should:

* **widen** what the reference under-samples. This capture is winter with LAI around 0.01, so
  give LAI, temperatures, radiation and wind physically plausible daytime ranges;
* **pin** what the scenario excludes. "Snow-free" means no snow, no ice and no surface water;
* **derive** what must be consistent with something else: potential temperature from air
  temperature, density from the ideal-gas law, sunlit and shaded LAI that sum to `elai`, roughness
  lengths from canopy height, and so on.

`derive` rules may use each other; SPEL orders them by dependency and rejects cycles. The next
cell replaces the stats ranges with such a scenario and adds the tolerance to the property.

In [ ]:
INPUTS = """
[scenario.snowfree_day.inputs]
# widen: a daytime, growing-season canopy
"top_as%tbot" = { range = [278.0, 313.0] }
"top_as%qbot" = { range = [0.001, 0.02] }
"top_as%ubot" = { range = [0.5, 10.0] }
"top_af%lwrad" = { range = [200.0, 450.0] }
"col_es%t_grnd" = { range = [278.0, 315.0] }
"col_es%emg" = { range = [0.94, 0.98] }
"col_pp%dz" = { range = [0.02, 1.5] }
"canopystate_vars%elai_patch" = { range = [0.1, 6.0] }
"canopystate_vars%esai_patch" = { range = [0.05, 1.5] }
"canopystate_vars%htop_patch" = { range = [0.3, 20.0], pft = { 13 = { range = [0.3, 1.0] } } }
"solarabs_vars%sabv_patch" = { range = [0.0, 600.0] }
"solarabs_vars%parsun_z_patch" = { range = [0.0, 1200.0] }
"solarabs_vars%parsha_z_patch" = { range = [0.0, 150.0] }
# pin: no snow, no ice, no surface water
"col_ws%h2osoi_ice" = 0.0
"col_ws%frac_sno_eff" = 0.0
"col_ws%snow_depth" = 0.0
"col_ws%frac_h2osfc" = 0.0

[scenario.snowfree_day.derive]
"soilstate_vars%rootfr_patch" = "1.0 / nlevgrnd"   # must sum to 1 over the soil layers
"top_as%thbot" = "top_as%tbot(t)"
"top_as%vbot" = "top_as%ubot(t)"
"top_as%rhobot" = "top_as%pbot(t) / (287.04 * top_as%tbot(t))"
"top_as%po2bot" = "0.209 * top_as%pbot(t)"
"top_as%pco2bot" = "4.0e-4 * top_as%pbot(t)"
"col_es%thv" = "forc_th(t) * (1.0 + 0.61 * forc_q(t))"
"col_es%t_h2osfc" = "t_grnd(c)"
"col_es%t_soisno" = "t_grnd(c)"
"col_ws%qg_soil" = "qg(c)"
"col_ws%qg_snow" = "qg(c)"
"col_ws%qg_h2osfc" = "qg(c)"
"veg_es%thm" = "forc_t(t) + 0.0098 * frictionvel_vars%forc_hgt_t_patch(p)"
"veg_es%emv" = "1.0 - exp(-(elai(p) + esai(p)))"
"veg_ws%fdry" = "(1.0 - fwet(p)) * elai(p) / (elai(p) + esai(p))"
"canopystate_vars%laisun_patch" = "0.5 * elai(p)"
"canopystate_vars%laisha_patch" = "elai(p) - laisun(p)"
"canopystate_vars%laisun_z_patch" = "laisun(p)"
"canopystate_vars%laisha_z_patch" = "laisha(p)"
"surfalb_vars%tlai_z_patch" = "elai(p)"
"canopystate_vars%displa_patch" = "0.67 * htop(p)"
"frictionvel_vars%z0mv_patch" = "0.055 * htop(p)"
"frictionvel_vars%forc_hgt_u_patch" = "30.0 + htop(p)"
"frictionvel_vars%forc_hgt_t_patch" = "30.0 + htop(p)"
"frictionvel_vars%forc_hgt_q_patch" = "30.0 + htop(p)"
"""
PROPS = PROPS.replace("rssun(p) <= 2.0e4", "rssun(p) <= 2.0e4 * (1.0 + 1.0e-12)") \
             .replace("rssha(p) <= 2.0e4", "rssha(p) <= 2.0e4 * (1.0 + 1.0e-12)")
sp = write_spec(HEADER, SCENARIO, INPUTS, PROPS)
print("derive order:", [v.split("%")[-1] for v, _ in sp.scenarios["snowfree_day"].derive][:8], "...")
table, outcomes = run(sp)
table

In [ ]:
show = ["veg_es%t_veg", "energyflux_vars%btran_patch", "energyflux_vars%btran2_patch",
        "veg_wf%qflx_tran_veg", "canopystate_vars%elai_patch"]
rng = np.random.default_rng(0)
with Session(sp, inv) as s:
    scen = sp.scenarios["snowfree_day"]
    elems = s.elements((scen.level, scen.filter))
    ex = s.inputs_at(scen, 0.5)
    mid = per_element(s, s.arrays(ex, s.compute(scen, ex, show[:-1])), show, elems)
    wet = []
    for _ in range(5):          # random draws: u uniform per entry
        ex = s.inputs_at(scen, rng.random)
        btran = s.compute(scen, ex, ["energyflux_vars%btran_patch"])["energyflux_vars%btran_patch"]
        wet.append(np.mean(np.asarray(btran.data)[elems - 1] > 0))
print(f"elements with btran > 0 in 5 random draws: {np.round(wet, 2)}")
mid.groupby("pft").first()

`btran` is still 0 at the scenario's midpoint, and so is the transpiration. Before calling
that a bug, follow it into the code. In `calc_root_moist_stress_clm45default`
(SoilMoistStressMod), a layer's root resistance factor comes from the matric potential of
its *liquid* saturation, `-sucsat * s**(-bsw)`, capped at the wilting point `smpsc`. At the
midpoint (`s` ≈ 0.34, `sucsat` ≈ 218 mm, `bsw` ≈ 7) that is about −490 kPa, drier than
`smpsc` = −255 kPa, so every layer is at wilting point and `btran = 0`. `btran2`, which uses
the total water content, is not 0. The midpoint is a dry soil, not a broken routine.

The random draws show what the tests actually see: there every element has `btran > 0`,
because each soil layer gets its own draw and some layer is always wet enough. The midpoint,
with every layer at the same dry value, is the exception. Had it been the other way round,
checks such as "transpiration is not negative" would hold trivially, because the scenario
would never reach transpiration. Coverage
questions like this one (*does the scenario reach the branch I want to test?*) come up for
every routine; a quick loop like the one above answers them. To test the stressed and
unstressed regimes separately, split the scenario as Part 8 does with `assume`.

## 8. Metamorphic relations

A property needs to know the right answer. A **relation** needs only to know how the answer
changes. Each relation runs the routine on *two* inputs in one call: the base sets get a drawn
example, and the follow-up sets get a copy of it, transformed by `follow`. `derive` rules are then
applied again, except to the transformed variables. `expect` compares the two runs per element
through `base%x` and `follow%x`.

Three relations for CanopyFluxes:

1. **Copies give identical outputs.** A relation without `follow`: the follow-up half is a copy.
   This is a cheap check for uninitialized memory, a wrong index, and the harness itself. Use it
   in every spec. `same(a, b)` is true when both are identical or both NaN.
2. **The order of the patches does not matter.** `permute = true` puts the copies in another
   order (among the patches of one pft). A result that depends on a neighbouring element passes
   the first relation, because both halves have the same neighbours, and fails this one.
3. **More absorbed shortwave does not cool the leaf.** Raising `sabv` adds energy to the leaf
   energy balance, so `t_veg` should not go down.

In [ ]:
RELS = """
[[relation]]
name = "copies of the inputs give identical outputs"
expect = "same(follow%t_veg(p), base%t_veg(p)) .and. same(follow%eflx_sh_veg(p), base%eflx_sh_veg(p))"

[[relation]]
name = "the order of the patches does not matter"
permute = true
expect = "same(follow%t_veg(p), base%t_veg(p)) .and. same(follow%qflx_tran_veg(p), base%qflx_tran_veg(p))"

[[relation]]
name = "more absorbed shortwave does not cool the leaf"
follow = { "solarabs_vars%sabv_patch" = "1.1 * solarabs_vars%sabv_patch(p) + 10.0" }
expect = "follow%t_veg(p) >= base%t_veg(p) - 1.0e-3"
"""
sp = write_spec(HEADER, SCENARIO, INPUTS, PROPS, RELS)
table, outcomes = run(sp, checks=["copies*", "the order*", "more absorbed*"], settings={"max_examples": 40})
table

In [ ]:
for o in outcomes:
    if not o.passed:
        lines = o.message.splitlines()
        keep = [l for l in lines if not l.startswith("    ") or any(k in l for k in
                ("%t_veg =", "tbot", "t_grnd", "ubot", "htop", "sabv", "elai"))]
        print("\n".join(keep[:30]))
        print("saved:", Path(o.failure).relative_to(case_dir.parent))

The shortwave relation fails, by a lot: the leaf with more absorbed shortwave ends about 2 K
*cooler*. Look at the shrunk example. The wind is at the low end of its rule (`ubot` = 0.5 m/s)
and the ground is far from the air temperature (`t_grnd` = 315 K under `tbot` = 291.5 K).
(While the example spec was written, Hypothesis found the same failure on a short grass patch
with the ground much *colder* than the air.) More absorbed shortwave changes the Monin–Obukhov
stability iteration (the Obukhov length, the aerodynamic resistances), and the leaf temperature
it converges to is lower.

Is it a scenario mistake, as in Part 7? Each input is inside a plausible range, though a 315 K
ground under 291 K air is an extreme pairing. The usual next step is to **separate the regime**:
find the conditions under which the relation should hold without question, keep it there, and
record the rest explicitly. `assume` drops elements that violate a condition (an example with no
element left is discarded), and `scenarios = [...]` restricts a check to some scenarios.

A first guess: a stable surface layer (ground colder than the air) is the problem. Test the
guess by keeping only elements whose ground is at least as warm as the air.

In [ ]:
UNSTABLE = (SCENARIO.replace("snowfree_day", "snowfree_day_unstable")
            + 'assume = ["t_grnd(c) >= forc_t(t)"]       # ground at least as warm as the air: no stable layer\n'
            + INPUTS.replace("snowfree_day", "snowfree_day_unstable"))
RELS_UNSTABLE = RELS.replace(
    'name = "more absorbed shortwave does not cool the leaf"',
    'name = "more absorbed shortwave does not cool the leaf"\nscenarios = ["snowfree_day_unstable"]')
sp = write_spec(HEADER, SCENARIO, INPUTS, UNSTABLE, PROPS, RELS_UNSTABLE)
table, outcomes = run(sp, scenarios=["snowfree_day_unstable"], checks=["more absorbed*"],
                      settings={"max_examples": 40})
table

In [ ]:
for o in outcomes:
    if not o.passed:
        lines = o.message.splitlines()
        keep = [l for l in lines if not l.startswith("    ") or any(k in l for k in
                ("%t_veg =", "tbot", "t_grnd", "ubot", "htop", "sabv", "elai"))]
        print("\n".join(keep[:30]))

The guess was wrong, and the test of it says so: the relation fails with a warm ground too.
What the two failing examples share is not the sign of the stability but the wind, `ubot` at
the low end of its range. Second guess: the relation needs some wind.

In [ ]:
WINDY = (SCENARIO.replace("snowfree_day", "snowfree_day_windy")
         + 'assume = ["top_as%ubot(t) >= 2.0"]         # at least a light breeze\n'
         + INPUTS.replace("snowfree_day", "snowfree_day_windy"))
RELS_SPLIT = RELS.replace(
    'name = "more absorbed shortwave does not cool the leaf"',
    'name = "more absorbed shortwave does not cool the leaf"\nscenarios = ["snowfree_day_windy"]')
sp = write_spec(HEADER, SCENARIO, INPUTS, WINDY, PROPS, RELS_SPLIT)
table, outcomes = run(sp, scenarios=["snowfree_day_windy"], checks=["more absorbed*"],
                      settings={"max_examples": 80})
table

With at least 2 m/s of wind the relation holds in these 80 examples. A pass is not a proof:
while this notebook was written, the same relation restricted to near-neutral conditions
(`abs(t_grnd(c) - forc_t(t)) <= 5.0`) failed once at 2.2 m/s. What can be said is this: more
absorbed shortwave can cool the leaf, by up to about 2 K in these examples, mostly in weak wind,
and the stability iteration is the place to look. That is a finding about the model, one to take
to a domain scientist. Record it in the spec: keep the relation where it is expected to hold,
and keep the failing form, with a comment and the regime, as a known failure (the example spec
does that).

A failing run saves the shrunk example under
`unit-tests/proptest-data/<CASE>/failures/<scenario>__<check>/`: the inputs in the format of a
capture, and `example.json`. `spel proptest replay <that directory>` runs it again without
Hypothesis, which is what you want while debugging or after a fix. Hypothesis also keeps failing
examples in `unit-tests/proptest-data/<CASE>/hypothesis/` and tries them first in the next run
(not when a `seed` is given, as in this notebook). Change `seed` (or drop it) to explore further,
and raise `max_examples` for a longer search.

Other relations worth writing for an energy-balance routine:

* **conservation**: compute a residual from the outputs and require `close(residual, 0, atol=...)`;
* **symmetry or invariance**: renaming pfts with equal parameters gives the same outputs;
* **monotonicity** in a forcing, as above. These are the most informative, but each needs its
  regime stated.

## 9. Namelist options: another setting as the follow-up

So far every run used one configuration, the reference's options plus `[namelist]`. Two things
can change that:

* a scenario with `namelist = { use_hydrstress = true }` runs under other options, with its own
  set of inputs and outputs;
* a relation with `namelist = { ... }` runs its follow-up under other options: the same example,
  another setting. `base%x` and `follow%x` are then the two runs.

Namelist options are module scalars, so two settings cannot share one call. SPEL starts a second
worker process on the same library for the other setting; no rebuild is needed, because the
options are constants the case reads at start.

Relations between settings come in a few kinds. An option that should not matter for these
outputs gives `same(...)`. Two settings that select the same code must agree exactly. Here
`force_land_gustiness` asks for the land's own gustiness whatever `atm_gustiness` says:

In [ ]:
RELS_NML = """
[[relation]]
name = "methane off: the canopy fluxes are identical"
scenarios = ["snowfree_day"]
namelist = { use_lch4 = false }
expect = "same(follow%t_veg(p), base%t_veg(p)) .and. same(follow%qflx_tran_veg(p), base%qflx_tran_veg(p))"

[[relation]]
name = "force_land_gustiness overrides atm_gustiness"
scenarios = ["snowfree_day"]
namelist = { atm_gustiness = true, force_land_gustiness = true }
expect = "same(follow%ustar(p), base%ustar(p)) .and. same(follow%t_veg(p), base%t_veg(p))"
"""
sp = write_spec(HEADER, SCENARIO, INPUTS, WINDY, PROPS, RELS_SPLIT, RELS_NML)
table, outcomes = run(sp, checks=["methane*", "force_land*"], settings={"max_examples": 40})
table

An option may bring its own inputs. With `use_c13` on, CanopyFluxes reads the fractionation
factors, the 13CO2 partial pressure and a down-regulation factor. The scenario has no rule for
them, because they are not inputs of its own configuration, and the run says so before it
starts:

In [ ]:
C13 = """
[[relation]]
name = "C13 on: the fluxes are identical and the 13C flux is finite"
scenarios = ["snowfree_day"]
namelist = { use_c13 = true }
expect = \"\"\"same(follow%t_veg(p), base%t_veg(p)) .and. same(follow%qflx_tran_veg(p), base%qflx_tran_veg(p))
            .and. finite(follow%photosyns_vars%c13_psnsun_patch(p))\"\"\"
"""
sp = write_spec(HEADER, SCENARIO, INPUTS, WINDY, PROPS, RELS_SPLIT, RELS_NML, C13)
try:
    run(sp, checks=["C13*"])
except spec.SpecError as exc:
    print("SpecError:", textwrap.fill(str(exc), 110))

Give them rules in the scenario. There they are **dormant**: drawn with every example, set only
in a run whose options read them. `follow%c13_psnsun_patch` is valid in this relation because
the name is checked against the follow-up's options; `base%c13_psnsun_patch` would be an error.

In [ ]:
INPUTS_C13 = (INPUTS.replace("[scenario.snowfree_day.inputs]\n", """[scenario.snowfree_day.inputs]
# only read with use_c13 on: dormant in the scenario's own runs
"photosyns_vars%alphapsnsun_patch" = 1.0
"photosyns_vars%alphapsnsha_patch" = 1.0
"cnstate_vars%downreg_patch" = { range = [0.0, 1.0] }
""") + '"top_as%pc13o2bot" = "0.011 * top_as%pco2bot(t)"\n')        # joins the derive table, the last one of INPUTS
sp = write_spec(HEADER, SCENARIO, INPUTS_C13, WINDY, PROPS, RELS_SPLIT, RELS_NML, C13)
scen = sp.scenarios["snowfree_day"]
print("dormant in snowfree_day:", spec.dormant_rules(scen, inv.configured(spec.constants(sp, inv, scen))))
table, outcomes = run(sp, checks=["C13*"], settings={"max_examples": 40})
table

The isotope code only adds outputs, as it should. Not every guess about an option survives:
while the example spec was written, "without the land's gustiness the friction velocity is not
higher" (`atm_gustiness = true`, `follow%ustar(p) <= base%ustar(p)`) failed in a few elements by
more than 1e-4, because the stability iteration takes another path. Triage such a failure like
any other (Part 7).

`spel proptest show <spec>` prints, for every scenario, the options with the source of each
value, how many variables they leave unused, and the dormant rules.

## 10. Coverage: does the scenario reach the code?

A check can pass because the scenario never runs the code the check is about. The question came
up in Part 7 (does the scenario reach transpiration?). `coverage.measure` (the command
`spel proptest coverage`) answers it for the whole call tree of the routine: it runs examples on
a gfortran debug build of the case (`-O0` with coverage counters, built next to the normal one),
reads the line counts with `gcov`, and lists the `if`, `else if` and `else` branches none of
whose lines ran. The branches come from SPEL's parser, not from the text. Branches the namelist
options turn off are counted apart: the scenario could not have reached them.

In [ ]:
cov = coverage.measure(sp, inv, "snowfree_day", examples=10)
ran = [r for r in cov.routines if r.calls > 0]
off = [b for b in cov.branches if b.off]
reachable = [b for b in cov.branches if not b.off]
print(f"{len(ran)} routines ran, {len(cov.routines) - len(ran)} of the call tree never did.")
print(f"{len(off)} branches are off under the namelist options, e.g.", sorted({b.condition for b in off})[:4])
print(f"{len(reachable)} branches could be reached and never were. In CanopyFluxesMod.F90:")
for b in reachable:
    if b.file == "CanopyFluxesMod.F90":
        print(f"  line {b.line}: {b.kind} ({b.condition[:70]})")
print("and, among those of PhotosynthesisMod.F90:")
for b in reachable:
    if b.file == "PhotosynthesisMod.F90" and any(k in b.condition for k in ("c3flag", "par_z", "laican")):
        print(f"  line {b.line}: {b.kind} ({b.condition[:70]})")

What is left is what the scenario itself decides: bare ground (`frac_veg_nosno == 0`, excluded
by the scenario), irrigation and crops (the domain has neither), and a few numerical safeguards.
Read the list for the surprises: a branch you meant to test and never reach needs another
scenario, another pft in `[domain]`, or a wider rule. The list for `PhotosynthesisMod.F90`
shows that this scenario never takes the C4 path (pfts 1, 4 and 13 are all C3), never has a dark
canopy layer (`par_z <= 0`), and no longer reaches the canopy without leaf area that made
Part 7's property fail (`laican > 0` is always true now).

## 11. Differential checks: two builds, one answer

SPEL's usual test compares a refactored routine with one capture, bit for bit. A
`[[differential]]` check makes the same comparison on every example a scenario draws: each
example runs on this case and on the one `[compare]` names, and the outputs must agree. The other
side is another *case* (`case = "cf_acc"`: a refactored copy of the routine) or another *build*
of this case (`build = { fc = "nvfortran", acc = "MULTICORE" }`), or both.

Here the other side is the debug build Part 10 made. With gfortran the two must give identical
numbers, so no tolerance is given; `outputs` defaults to every output the routine has under the
scenario's namelist options (101 here).

In [ ]:
DIFF = """
[compare]
build = { dbg = true }            # the same case, the gfortran debug build

[[differential]]
name = "the debug build gives the same answers"
scenarios = ["snowfree_day"]
"""
sp = write_spec(HEADER, SCENARIO, INPUTS_C13, WINDY, PROPS, RELS_SPLIT, RELS_NML, C13, DIFF)
table, outcomes = run(sp, checks=["the debug build*"], settings={"max_examples": 20})
table

Across compilers the answers differ by rounding, so give a tolerance
(`outputs = ["t_veg", ...]`, `rtol = 1.0e-9`); for an OpenACC build against the serial one, a
difference larger than that usually means a race. `expect = "..."` replaces the default
comparison with any expression of `ref%x` (this case) and `dev%x` (the other one).

## 12. Active subspace: which inputs matter

With ~100 ruled inputs it helps to know which ones drive an output. `subspace.analyze` estimates
the gradient of an output with respect to each input, normalized to [-1, 1], by forward
differences: one input direction per domain copy, so one routine call computes many directions.
It then takes the eigendecomposition of the averaged outer product of the gradients:

* **activity scores** rank the inputs;
* **the eigenvalue gap** says how many directions matter.

Use it to choose what to vary in relations, or which inputs deserve careful rules. It runs on any
scenario. `inputs=` restricts it to a few inputs for a quick look, as below; omit it for all of
them. `parameters=True` also steps ruled parameters (one more call per parameter).

In [ ]:
sub = subspace.analyze(sp, inv, "snowfree_day", "t_veg(p)", samples=2, bootstrap=50, inputs=[
    "top_as%tbot", "col_es%t_grnd", "top_as%ubot", "top_as%qbot", "solarabs_vars%sabv_patch",
    "canopystate_vars%elai_patch", "canopystate_vars%htop_patch", "veg_ws%fwet"])
json_path, md_path = subspace.write(sub, case_dir)
from IPython.display import Markdown
Markdown(subspace.report_md(sub, top=8))

## 13. The same from the command line

Each step above has a CLI equivalent. The spec file this notebook wrote can be used directly:

```bash
T=unit-tests/proptest-data/canflux/tutorial/tutorial.proptest.toml
spel proptest init     canflux --namelist soil_stress_method=0   # a starter spec, its [namelist] table
spel proptest show     canflux --kind input                     # inventory
spel proptest show     $T                                      # scenarios: namelist options, missing and dormant rules
spel proptest probe    $T -s snowfree_day                      # what a few examples do
spel proptest run      $T -n 50 -k "soil water*" [--json out.json] [--junit out.xml]
spel proptest replay   unit-tests/proptest-data/canflux/failures/<scenario>__<check>
spel proptest coverage $T -s snowfree_day
spel proptest subspace $T --scenario snowfree_day --output "t_veg(p)"
```

`run` exits with 0 when every check passes, 1 when one fails and 2 when the spec has a problem
or no check is selected, so it fits CI. The finished version of this spec is
`spel/proptest/examples/canopyfluxes.proptest.toml`, and the format is described in
`spel/proptest/SPEC.md`.

## 14. Your own subroutine: a checklist

1. **Extract it.** Run `spel create -s <routine> -c <case>` (add `--reanalyze` after editing
   E3SM). If you want stats, capture reference data too (README §2). A capture from a
   different season or site than the one you care about is still useful for constants and
   extents.
2. **Start a spec.** `spel proptest init <case>` builds the library, writes the rule file and a
   starter spec with a `[namelist]` table and `# TODO` lines. Decide the options first: they
   decide which variables the spec is about (Part 2). A build failure here is usually a case
   problem; check that `spel run <case>` works.
3. **Read the inventory** (Part 2). For each `input`/`inout`, ask: what is a physically
   meaningful range for the regime I want to test? Check that the `output` list contains no
   variable you know the routine reads first. If it does, the classification is wrong; report
   it rather than ruling around it.
4. **Go through the rules** (Parts 3 and 7): widen the under-sampled ones, pin what the scenario
   excludes, and `derive` everything that must be consistent. Typical candidates are potential
   temperature, density, partial pressures, sunlit/shaded splits, fractions that must sum to 1,
   and roughness lengths from heights. `spel proptest show <spec>` lists what still has no rule.
5. **Pick the domain.** Choose pfts that cover the parameter variety (a tree, a grass, a crop if
   relevant), `per_pft` ≥ 2, and an even `nsets`. Constants the reference lacks go in
   `[domain.constants]`; the run tells you which.
6. **Probe it** (Part 6): `spel proptest probe <spec>`. Look at what is never written before
   writing checks, and at `spel proptest coverage <spec>` (Part 10) for the branches the scenario
   never enters.
7. **Write properties**: start with finiteness and ranges with a physical meaning (fractions in
   [0, 1], resistances > 0, fluxes with a known sign).
8. **Write relations**: copies-identical and a permuted copy first, then monotonicities and
   conservation laws, each with its regime stated (use `assume` and per-scenario checks).
9. **Triage every failure**: model, scenario or property? Fix scenarios with rules and `derive`;
   record model findings in the spec, with a comment and the regime. `spel proptest replay` runs
   the saved example again.
10. **Compare namelist settings** (Part 9): options that should not matter, settings that select
    the same code, invariants that must hold under either setting. Give the inputs only the
    other setting reads their rules.
11. **Compare builds** (Part 11) when you refactor the routine or port it to OpenACC: a
    `[[differential]]` check against the original case over the same scenarios.
12. **Use the subspace** to see which inputs drive the outputs you care about.

**Pitfalls seen so far**

* Inconsistent constants: a fraction that sums to 1.0000005 (Part 7).
* A constant the routine reads but the capture doesn't have stops the domain build. Set it in
  `[namelist]` (or `[domain.constants]`). A parameter the capture lacks stops the run; give it a
  rule.
* An option that is on in no capture may need inputs, dimensions or parameters the reference
  never recorded (`use_hydrstress` does here); the run names each one.
* Fill values (1e36) in the reference produce absurd stats ranges; `finite()` treats
  |x| ≥ 1e35 as not finite.
* Parameters (pftcon) are fixed at the reference's values unless you rule them with `scale`.
* Comparing a result exactly with a bound the code reaches is fragile (Part 7): give the bound a
  relative tolerance.
* In a check, the name of a variable the routine both reads and writes is its value *after* the
  run; `input%x` is the value before.
* Check names can contain `[...]`. Selection tries the exact name before treating it as a glob.
* In a script (not a notebook), put the calls under `if __name__ == "__main__":`. The worker is a
  spawned process.

**Exercise.** Pick another routine from `elm_drv`'s tree, for example the bare-ground fluxes
(`BareGroundFluxes`) or the soil-surface fluxes (`SoilFluxes`). Copy this notebook, change `CASE`
and `REFERENCE`, and work through Parts 1–8. A good first relation for `SoilFluxes` is
conservation: the energy entering the surface equals the sum of the fluxes leaving it.